# 10 — การสร้างกราฟด้วย Matplotlib

**คอร์ส Python for Geography** · ภาควิชาภูมิศาสตร์ ชั้นปีที่ 2
**วันที่ 6 · ช่วงที่ 1 · เวลาโดยประมาณ 60 นาที**

---

### สิ่งที่จะได้จากบทนี้
- สร้างกราฟ **line / bar / scatter / histogram** และใส่องค์ประกอบให้ครบ
- จัดหลายกราฟในภาพเดียวด้วย `subplots()`
- ตั้งค่า **ฟอนต์ไทย** ให้แสดงผลถูกต้องใน Colab
- พล็อตตำแหน่งสถานีบนระนาบพิกัด — ก้าวแรกสู่การทำแผนที่

> **วิธีเรียน:** โค้ดในเซลล์สีเทาคือโค้ดที่อาจารย์จะรันให้ดู
> ใต้แต่ละหัวข้อจะมีเซลล์ ✍️ ว่างไว้ — **ให้พิมพ์ตามด้วยตัวเอง ห้าม copy–paste**
> การพิมพ์เองคือวิธีเดียวที่จะจำ syntax ได้

## 0. เตรียมไฟล์ข้อมูล

In [ ]:
# ===== เตรียมไฟล์ข้อมูลสำหรับบทนี้ (รันครั้งเดียวต่อ session) =====
# ถ้าอาจารย์เปลี่ยนชื่อ repo ให้แก้ URL บรรทัดล่างนี้บรรทัดเดียว
BASE = "https://raw.githubusercontent.com/jamorn12/Teaching_Python_for_Geography/main/"

import os, subprocess, sys, urllib.request

FILES = ["stations.csv", "rainfall_daily_2025.csv", "messy_rainfall.csv"]
os.makedirs("data", exist_ok=True)


def have_all():
    return all(os.path.exists(os.path.join("data", n)) for n in FILES)


# 1) ถ้ามีไฟล์อยู่แล้ว (เช่น clone repo มา หรืออัปโหลดเอง) ก็ใช้ได้เลย
if not have_all():
    # 2) ลองดาวน์โหลดจาก GitHub
    for name in FILES:
        target = os.path.join("data", name)
        if not os.path.exists(target):
            try:
                urllib.request.urlretrieve(BASE + "data/" + name, target)
            except Exception:
                pass

if not have_all():
    # 3) สำรองสุดท้าย: สร้างข้อมูลชุดเดียวกันขึ้นมาใหม่จาก make_datasets.py
    script = None
    for cand in ["make_datasets.py", "utils/make_datasets.py"]:
        if os.path.exists(cand):
            script = cand
            break
    if script is None:
        try:
            urllib.request.urlretrieve(BASE + "utils/make_datasets.py", "make_datasets.py")
            script = "make_datasets.py"
        except Exception:
            script = None
    if script:
        subprocess.run([sys.executable, script, "--outdir", "data"], check=True)

if have_all():
    print("ข้อมูลพร้อมใช้งาน:", sorted(os.listdir("data")))
else:
    print("ยังไม่พบข้อมูล -> ให้ upload โฟลเดอร์ data/ เข้ามา "
          "หรือรัน utils/make_datasets.py เพื่อสร้างข้อมูลใหม่")

## 1. ตั้งค่าฟอนต์ไทย

matplotlib ไม่มีฟอนต์ไทยมาให้ ถ้าไม่ตั้งค่า ตัวอักษรไทยจะกลายเป็นสี่เหลี่ยม (□□□)
เซลล์นี้ดาวน์โหลดฟอนต์ Sarabun มาติดตั้ง ถ้าอินเทอร์เน็ตมีปัญหาก็ยังพล็อตได้ แต่ให้ใช้ label ภาษาอังกฤษแทน

In [ ]:
import os, urllib.request
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

THAI_FONT_OK = False
try:
    url = "https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf"
    path = "Sarabun-Regular.ttf"
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    fm.fontManager.addfont(path)
    matplotlib.rc("font", family="Sarabun")
    matplotlib.rcParams["axes.unicode_minus"] = False
    THAI_FONT_OK = True
    print("ฟอนต์ไทยพร้อมใช้งาน")
except Exception as exc:
    print("ตั้งฟอนต์ไทยไม่สำเร็จ:", exc, "-> ใช้ label ภาษาอังกฤษแทน")

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110

## 2. กราฟเส้น (line plot) — ข้อมูลตามเวลา

In [ ]:
import pandas as pd

df = pd.read_csv("data/rainfall_daily_2025.csv")
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month

chb = df[df["station_id"] == "CHB01"].sort_values("date")

plt.plot(chb["date"], chb["rain_mm"])
plt.title("ฝนรายวัน สถานีชลบุรี ปี 2025")
plt.xlabel("วันที่")
plt.ylabel("ปริมาณฝน (mm)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### ใส่หลายเส้น ตั้งค่าเส้น และใส่ legend

องค์ประกอบที่กราฟวิชาการ **ต้องมีเสมอ**: ชื่อกราฟ, ชื่อแกนพร้อมหน่วย, legend, และแหล่งข้อมูล

In [ ]:
monthly = df.groupby(["station_id", "month"])["rain_mm"].sum().reset_index()

for sid in ["CHB01", "SRC01", "NYI01", "STH01"]:
    sub = monthly[monthly["station_id"] == sid]
    plt.plot(sub["month"], sub["rain_mm"], marker="o", label=sid)

plt.title("ฝนรวมรายเดือน 4 สถานี จังหวัดชลบุรี ปี 2025")
plt.xlabel("เดือน")
plt.ylabel("ปริมาณฝนรวม (mm)")
plt.xticks(range(1, 13))
plt.legend(title="สถานี")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 3. กราฟแท่ง (bar chart) — เปรียบเทียบกลุ่ม

In [ ]:
annual = df.groupby("station_id")["rain_mm"].sum().round(1)

bars = plt.bar(annual.index, annual.values, color="#3b7dd8", edgecolor="black", linewidth=0.6)
plt.title("ฝนรวมทั้งปี 2025 รายสถานี")
plt.ylabel("ปริมาณฝนรวม (mm)")
plt.grid(axis="y", alpha=0.3)

# เขียนตัวเลขบนแท่ง
for b, v in zip(bars, annual.values):
    plt.text(b.get_x() + b.get_width()/2, v + 15, f"{v:.0f}", ha="center", fontsize=9)

plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 4. Histogram — การกระจายของข้อมูล

ฝนรายวันมีการแจกแจงแบบ **เบ้ขวา (right-skewed)** มาก คือวันไม่มีฝนเยอะมาก
และมีวันฝนหนักไม่กี่วันที่ค่าสูงลิ่ว การดู histogram ช่วยให้เลือกสถิติได้ถูก
(ค่าเฉลี่ยอาจไม่เหมาะกับข้อมูลแบบนี้ ควรดู median และ percentile ด้วย)

In [ ]:
rain_only = df.loc[df["rain_mm"] >= 0.1, "rain_mm"]

plt.hist(rain_only, bins=30, color="#6aa84f", edgecolor="black", linewidth=0.5)
plt.title("การกระจายของปริมาณฝนรายวัน (เฉพาะวันที่มีฝน)")
plt.xlabel("ปริมาณฝน (mm)")
plt.ylabel("จำนวนวัน")
plt.grid(axis="y", alpha=0.3)
plt.show()

print(rain_only.describe().round(2))
print("เปอร์เซ็นไทล์ที่ 95 =", round(rain_only.quantile(0.95), 1), "mm")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 5. Scatter plot — ตำแหน่งสถานีบนระนาบพิกัด

พล็อต `lon` บนแกน x และ `lat` บนแกน y แล้วให้ **ขนาดจุด** แทนปริมาณฝน
นี่คือหลักการเดียวกับ proportional symbol map ในวิชาการทำแผนที่
ข้อสำคัญคือต้องสั่ง `axis("equal")` ไม่อย่างนั้นรูปร่างเชิงพื้นที่จะบิดเบี้ยว

In [ ]:
stations = pd.read_csv("data/stations.csv")
annual = df.groupby("station_id")["rain_mm"].sum().reset_index()
gdf = stations.merge(annual, on="station_id")

plt.figure(figsize=(7, 6))
sc = plt.scatter(gdf["lon"], gdf["lat"],
                 s=gdf["rain_mm"] / 3,
                 c=gdf["rain_mm"], cmap="Blues",
                 edgecolor="black", linewidth=0.8)

for _, r in gdf.iterrows():
    plt.text(r["lon"] + 0.02, r["lat"] + 0.02, r["station_id"], fontsize=9)

plt.colorbar(sc, label="ฝนรวมทั้งปี (mm)")
plt.title("ตำแหน่งสถานีและปริมาณฝนรวมปี 2025")
plt.xlabel("Longitude (°E)")
plt.ylabel("Latitude (°N)")
plt.axis("equal")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 6. `subplots()` — หลายกราฟในภาพเดียว

`fig, ax = plt.subplots(nrows, ncols)` แล้วสั่งงานผ่าน `ax[i]`
รูปแบบนี้คือวิธีที่ใช้จริงในงานตีพิมพ์ เพราะควบคุมได้ละเอียดกว่า `plt.` ตรง ๆ

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
station_ids = ["CHB01", "SRC01", "NYI01", "STH01"]

for ax, sid in zip(axes.flat, station_ids):
    sub = monthly[monthly["station_id"] == sid]
    ax.bar(sub["month"], sub["rain_mm"], color="#3b7dd8")
    ax.set_title(sid)
    ax.set_xticks(range(1, 13))
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("ฝนรวมรายเดือน ปี 2025 เปรียบเทียบ 4 สถานี", fontsize=14)
fig.supxlabel("เดือน")
fig.supylabel("ปริมาณฝน (mm)")
fig.tight_layout()
plt.show()

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 7. บันทึกรูปเป็นไฟล์

งานตีพิมพ์ควรใช้ `dpi=300` ขึ้นไป และ `bbox_inches="tight"` เพื่อไม่ให้ขอบโดนตัด

In [ ]:
fig.savefig("monthly_rainfall_4stations.png", dpi=300, bbox_inches="tight")
!ls -lh monthly_rainfall_4stations.png

## 8. แบบฝึกหัดท้ายบท

1. พล็อตกราฟเส้นฝนรายวันของสถานี `NYI01` เฉพาะเดือนกันยายน พร้อมชื่อกราฟและชื่อแกนครบ
2. ทำ bar chart แสดง **จำนวนวันฝนตก** (≥ 0.1 mm) รายเดือนของสถานี `CHB01`
3. ทำ scatter แสดงความสัมพันธ์ระหว่างฝนรายวันของ `CHB01` กับ `NYI01` ในวันเดียวกัน
   (ใบ้: ใช้ `pivot` หรือ `merge`) แล้วบอกว่าเห็นความสัมพันธ์แบบใด
4. รวมกราฟข้อ 1–3 ไว้ในภาพเดียวด้วย `subplots(1, 3)` แล้ว save เป็น PNG 300 dpi

In [ ]:
# ข้อ 1

In [ ]:
# ข้อ 2

In [ ]:
# ข้อ 3

In [ ]:
# ข้อ 4

### เฉลย
```python
# ข้อ 1
sep = df[(df["station_id"] == "NYI01") & (df["month"] == 9)].sort_values("date")
plt.plot(sep["date"], sep["rain_mm"], marker=".")
plt.title("ฝนรายวัน สถานีหนองใหญ่ เดือนกันยายน 2025")
plt.xlabel("วันที่"); plt.ylabel("ปริมาณฝน (mm)")
plt.xticks(rotation=45); plt.grid(alpha=0.3); plt.show()

# ข้อ 2
chb = df[df["station_id"] == "CHB01"].copy()
chb["is_rain"] = chb["rain_mm"] >= 0.1
days = chb.groupby("month")["is_rain"].sum()
plt.bar(days.index, days.values, color="#6aa84f")
plt.title("จำนวนวันฝนตกรายเดือน สถานีชลบุรี ปี 2025")
plt.xlabel("เดือน"); plt.ylabel("จำนวนวัน"); plt.xticks(range(1, 13))
plt.grid(axis="y", alpha=0.3); plt.show()

# ข้อ 3
pivot = df.pivot_table(index="date", columns="station_id", values="rain_mm")
plt.scatter(pivot["CHB01"], pivot["NYI01"], s=12, alpha=0.5)
plt.xlabel("CHB01 (mm)"); plt.ylabel("NYI01 (mm)")
plt.title("ความสัมพันธ์ฝนรายวันระหว่างสองสถานี")
plt.grid(alpha=0.3); plt.show()
print("สหสัมพันธ์ r =", round(pivot["CHB01"].corr(pivot["NYI01"]), 3))
# ตีความ: จุดกระจุกใกล้ศูนย์เพราะวันไม่มีฝนเยอะ ค่า r เป็นบวกอ่อน ๆ
# แปลว่าฝนในพื้นที่นี้เป็นแบบ convective คือตกเป็นหย่อม ไม่ได้ตกพร้อมกันทั้งจังหวัด

# ข้อ 4
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(sep["date"], sep["rain_mm"], marker=".")
ax[0].set_title("NYI01 ก.ย. 2025"); ax[0].tick_params(axis="x", rotation=45)
ax[1].bar(days.index, days.values, color="#6aa84f")
ax[1].set_title("วันฝนตกรายเดือน CHB01")
ax[2].scatter(pivot["CHB01"], pivot["NYI01"], s=10, alpha=0.5)
ax[2].set_title("CHB01 vs NYI01")
fig.tight_layout()
fig.savefig("exercise_day6.png", dpi=300, bbox_inches="tight")
plt.show()
```